# 📊 UPI Remitter Bank Reliability Analysis: Phase 2, Data Loader

🎯 **Goal:** Turn 24 messy NPCI monthly files into one clean table.

📥 **Input:** 24 Excel files in `raw/` (Top 50 remitter banks, Aug 2024 to Jul 2026).

📤 **Output:** `upi_remitter_clean.csv`, 1,200 rows (24 months x 50 banks).

🪜 **Steps:**
1. Load the raw files
2. Read the month from cell A1
3. Convert the month text into a date, and stop on duplicates
4. Clean the number columns
5. Load one file into a table
6. Load all 24 files and check the total
7. Save the CSV

## Step 1: Load the raw files into Colab
**What:** Upload `raw.zip` (the 24 NPCI files) and unzip it into a `raw/` folder.
**Check:** the last line should print 24.

In [2]:
from google.colab import files
files.upload()
!unzip -q -o raw.zip -d raw
!ls raw | wc -l

Saving raw.zip to raw.zip
24


## Step 2: Read the month from cell A1 of each file
**What:** Open every file and print the title in cell A1, for example `Top 50 Members (Jul'26)`.

**Why:** The month selected on the NPCI site can differ from the file's real month (this happened with Apr'25 and Jul'25).

**Why openpyxl:** pandas skips the title row, and we need it. We also read the first sheet by position, because sheet names differ across files.


In [3]:
import openpyxl, glob

for f in sorted(glob.glob('raw/*.xlsx')):
    ws = openpyxl.load_workbook(f, data_only=True).worksheets[0]
    print(f, '|', ws['A1'].value)

raw/upi_2024_08.xlsx | UPI Remitter Members - Top 50 Members (Aug'24)
raw/upi_2024_09.xlsx | UPI Remitter Members - Top 50 Members (Sep'24)
raw/upi_2024_10.xlsx | UPI Remitter Members - Top 50 Members (Oct'24)
raw/upi_2024_11.xlsx | UPI Remitter Members - Top 50 Members (Nov'24)
raw/upi_2024_12.xlsx | UPI Remitter Members - Top 50 Members (Dec'24)
raw/upi_2025_01.xlsx | UPI Remitter Members - Top 50 Members (Jan'25)
raw/upi_2025_02.xlsx | UPI Remitter Members - Top 50 Members (Feb'25)
raw/upi_2025_03.xlsx | UPI Remitter Members - Top 50 Members (Mar'25)
raw/upi_2025_04.xlsx | UPI Remitter Members - Top 50 Members (Apr'25)
raw/upi_2025_05.xlsx | UPI Remitter Members - Top 50 Members (May'25)
raw/upi_2025_06.xlsx | UPI Remitter Members - Top 50 Members (Jun'25)
raw/upi_2025_07.xlsx | UPI Remitter Members - Top 50 Members (Jul'25)
raw/upi_2025_08.xlsx | UPI Remitter Members - Top 50 Members (Aug'25)
raw/upi_2025_09.xlsx | UPI Remitter Members - Top 50 Members (Sep'25)
raw/upi_2025_10.xlsx

## Step 3: Convert the A1 title into a real date

**What:** A function that turns `(Jul'26)` into 2026-07-01, and stops the loader if two files have the same month.

**Why:** SQL and Power BI need a proper date to sort and chart by month, not text like "Jul'26".

In [4]:
import re
from datetime import date

MONTHS = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
          'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}

def month_from_title(title):
    m = re.search(r"\((\w{3})'(\d{2})\)", title)
    if not m:
        raise ValueError(f"No month found in title: {title}")
    mon, yy = m.groups()
    return date(2000 + int(yy), MONTHS[mon], 1)

seen = {}
for f in sorted(glob.glob('raw/*.xlsx')):
    ws = openpyxl.load_workbook(f, data_only=True).worksheets[0]
    d = month_from_title(ws['A1'].value)
    if d in seen:
        raise ValueError(f"Duplicate month {d}: {f} and {seen[d]}")
    seen[d] = f

print(len(seen), 'unique months:', min(seen), 'to', max(seen))

24 unique months: 2024-08-01 to 2026-07-01


## Step 4: Clean the number columns

**Columns:** Total Volume (Mn), Approved %, BD %, TD %.

**Columns we skip:** Sr. No. (changes every month, not useful), Debit Reversal Count and Debit Reversal Success % (extreme on tiny counts, **not part of our metrics**).

**What:** One function, `to_number`, that turns any cell value into a plain number.
*For example*, `'4,052.31'` (volume stored as text with a comma), `'94.40%'` (percentage as text), `' 09.11% '` (percentage with stray spaces) and `0.9122` (a fraction, already a number).

**Why:** The same column is stored differently in different months. One function means every file is cleaned the same way, and the tests show it works before we trust it.


In [7]:
# Look at the raw data
for name in ['upi_2024_08', 'upi_2025_01']:
    ws = openpyxl.load_workbook(f'raw/{name}.xlsx', data_only=True).worksheets[0]
    print('\n', name)
    for row in ws.iter_rows(min_row=2, max_row=7, max_col=8, values_only=True):
        print(row)


 upi_2024_08
('Sr. No.', 'upi_remitter_banks', 'Total Volume (In Mn)', 'Approved %', 'BD %', 'TD%', 'Total Debit Reversal Count (In Mn)', 'Debit Reversal Success %')
('1', 'State Bank Of India', '4,052.31', '94.40%', '5.39%', '0.21%', '11.86', '98.58%')
('2', 'HDFC Bank Ltd.', '1,270.59', '95.28%', '4.66%', '0.06%', '3.61', '99.34%')
('3', 'Bank of Baroda', '985.86', '92.45%', '6.95%', '0.60%', '3.26', '95.42%')
('4', 'Union Bank Of India', '941.93', '93.51%', '5.83%', '0.65%', '3.91', '68.43%')
('5', 'Punjab national Bank', '835.00', '91.99%', '7.92%', '0.09%', '2.96', '95.13%')

 upi_2025_01
('Sr. No.', 'UPI Remitter Banks', 'Total Volume (In Mn)', 'Approved %', 'BD %', 'TD%', 'Total Debit Reversal Count (In Mn)', 'Debit Reversal Success %')
(1, 'State Bank of India', 4664.49, 0.9122, 0.0793, 0.0084, 15.63, 0.9706)
(2, 'HDFC Bank Ltd.', 1419.33, 0.9364, 0.0628, 0.0008, 4.12, 0.9891)
(3, 'Bank of Baroda', 1159.99, 0.9122, 0.085, 0.0028, 3.31, 0.9733)
(4, 'Union Bank of India', 1086.2

In [8]:
def to_number(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).replace('\xa0', ' ').strip().replace(',', '').replace('%', '')
    return float(s)

assert to_number('4,052.31') == 4052.31
assert to_number('94.40%') == 94.4
assert to_number(' 09.11% ') == 9.11
assert to_number('0.21%\xa0') == 0.21
assert to_number(0.9122) == 0.9122
assert to_number(None) is None
print('all tests passed')

all tests passed


## Step 5: Load one file into a clean table

**What we are doing:**
We write a function called `load_file`. You give it one Excel file, and it gives back a clean table with 50 rows (one per bank).

**Columns in the table:**
month, bank name, volume, approved %, BD %, TD %

**Why:**
Right now each file looks different. Once this works on one file, we use it on all 24.

**One problem it solves:**
Some files store 0.91 and some store 91%. These mean the same thing. The function checks that approved + BD + TD adds up to 100 and fixes the scale if needed. If it can't, it stops.

In [9]:
import pandas as pd

def load_file(path):
    ws = openpyxl.load_workbook(path, data_only=True).worksheets[0]
    month = month_from_title(ws['A1'].value)
    rows = []
    for r in ws.iter_rows(min_row=3, max_col=6, values_only=True):
        if r[1] is None:
            continue
        bank = ' '.join(str(r[1]).replace('\xa0', ' ').split())
        rows.append({'month': month, 'bank_raw': bank,
                     'volume_mn': to_number(r[2]),
                     'approved_pct': to_number(r[3]),
                     'bd_pct': to_number(r[4]),
                     'td_pct': to_number(r[5])})
    df = pd.DataFrame(rows)

    pct = ['approved_pct', 'bd_pct', 'td_pct']
    total = df[pct].sum(axis=1)
    if ((total - 100).abs() <= 0.1).all():
        pass
    elif ((total - 1).abs() <= 0.001).all():
        df[pct] = df[pct] * 100
    else:
        raise ValueError(f"{path}: approved + BD + TD does not add to 100 or 1")
    return df

In [10]:
for name in ['upi_2024_08', 'upi_2025_01', 'upi_2026_07']:
    df = load_file(f'raw/{name}.xlsx')
    dev = (df[['approved_pct', 'bd_pct', 'td_pct']].sum(axis=1) - 100).abs().max()
    print(name, '| rows', len(df), '| volume', round(df.volume_mn.sum()), '| max sum deviation', round(dev, 3))
display(df.head())

upi_2024_08 | rows 50 | volume 15499 | max sum deviation 0.01
upi_2025_01 | rows 50 | volume 18005 | max sum deviation 0.01
upi_2026_07 | rows 50 | volume 25898 | max sum deviation 0.01


,month,bank_raw,volume_mn,approved_pct,bd_pct,td_pct
0,2026-07-01,State Bank of India,6622.022189,90.29,9.02,0.68
1,2026-07-01,Bank of Baroda,1774.429220,85.83,13.95,0.22
2,2026-07-01,HDFC Bank Ltd.,1737.216436,93.00,6.99,0.01
3,2026-07-01,Union Bank of India,1544.225883,89.60,10.39,0.01
4,2026-07-01,Punjab National Bank,1470.120376,88.20,11.10,0.70


## Step 6: Load all 24 files into one table

**What we are doing:**
Run `load_file` on every file and stack the results into one big table.

**Why:**
This is the table we will load into MySQL. Before using it we check it is complete: 24 months x 50 banks = 1,200 rows, no empty cells, no repeated bank in a month.

**Extra check:**
We add up each month's volume and failed payments, and compare with the numbers we already confirmed by hand.

In [11]:
frames = []
for f in sorted(glob.glob('raw/*.xlsx')):
    frames.append(load_file(f))
all_df = pd.concat(frames, ignore_index=True)

print('rows:', len(all_df))
print('months:', all_df.month.nunique())
print('rows per month:', sorted(all_df.groupby('month').size().unique()))
print('missing values:', int(all_df.isna().sum().sum()))
print('repeated bank in a month:', int(all_df.duplicated(['month', 'bank_raw']).sum()))

rows: 1200
months: 24
rows per month: [np.int64(50)]
missing values: 0
repeated bank in a month: 2


In [12]:
all_df['est_failed_mn'] = all_df.volume_mn * all_df.td_pct / 100

summary = all_df.groupby('month').agg(
    volume_mn=('volume_mn', 'sum'),
    est_failed_mn=('est_failed_mn', 'sum'))
summary['weighted_td_pct'] = summary.est_failed_mn / summary.volume_mn * 100
display(summary.round(3))

,volume_mn,est_failed_mn,weighted_td_pct
month,,,
2024-08-01,15498.84,54.888,0.354
2024-09-01,15591.75,76.423,0.490
2024-10-01,17345.81,116.384,0.671
2024-11-01,16219.87,75.899,0.468
2024-12-01,17586.04,85.836,0.488
2025-01-01,18004.92,90.542,0.503
2025-02-01,17048.29,71.511,0.419
2025-03-01,19324.05,84.499,0.437
2025-04-01,18881.74,71.563,0.379


## Step 7: Save the table as a CSV

**What we are doing:**
Save the 1,200-row table as `upi_remitter_clean.csv`, then read it back to make sure it saved correctly.

**Why:**
MySQL will load this file

**Note:**
We save only the 6 raw columns. Estimated failed payments will be calculated again in SQL, so we can rebuild every metric ourselves from the base data.

In [15]:
cols = ['month', 'bank_raw', 'volume_mn', 'approved_pct', 'bd_pct', 'td_pct']
all_df[cols].to_csv('upi_remitter_clean.csv', index=False)

check = pd.read_csv('upi_remitter_clean.csv')
print(check.shape)
display(check.head(3))

from google.colab import files
files.download('upi_remitter_clean.csv')

(1200, 6)


,month,bank_raw,volume_mn,approved_pct,bd_pct,td_pct
0,2024-08-01,State Bank Of India,4052.31,94.40,5.39,0.21
1,2024-08-01,HDFC Bank Ltd.,1270.59,95.28,4.66,0.06
2,2024-08-01,Bank of Baroda,985.86,92.45,6.95,0.60


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ✅ Summary: what this notebook did

- 📅 Loaded 24 files and took each month from cell A1, not the file name.
- 🧹 Cleaned three different number formats into one.
- 🔢 Checked approved + BD + TD = 100 on every row.
- 📈 Result: 1,200 rows, no missing values, monthly totals match my manual checks.

⚠️ **Open issue (for Phase 3):** Slice Small Finance Bank appears twice in Mar'26 and Jun'26 with different numbers. We keep both rows here and merge them in the bank mapping step.

🚧 **Not done yet:** bank names are still as NPCI wrote them (spelling and renames are fixed in Phase 3).

➡️ **Next:** Phase 3, bank mapping and loading into MySQL.